In [3]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as nb
from config import *
from utils import *
import json
import re
import datetime as dt
from os import listdir
from os.path import isfile, join
import ast


ModuleNotFoundError: No module named 'config'

In [5]:
cols = ['encounter_id','icd_primary_code','icd_primary_description', 'case_management_type','case_management_type_description','icd_secondaray_code','text_secondary_icd_official',
'icd_parent_code','icd_primary_pred','icd_secondary_pred', 'icd_coding_text', 'icd_coding_list','clinical_report']

In [8]:
training_size = 10000
val_size = 2000
test_size = 1000
job_name_train = "sample_10000_20250924"
job_name_val = "sample_5000_20250919"

In [9]:
df_train = prepare_training_files(path_results,job_name_train)

In [10]:
df_val = prepare_training_files(path_results,job_name_val)

In [11]:
if len(df_train) < training_size :
    nb_samples_from_val = training_size-len(df_train)
    df_train = pd.concat([df_train[cols], df_val.iloc[:nb_samples_from_val,:][cols]],axis=0)
else :
    nb_samples_from_val = 0

In [12]:
file_name_finetune = "train"
df_train.to_csv(path_results + job_name_train +"/"+file_name_finetune+"_v0.csv")
print(path_results + job_name_train +"/"+file_name_finetune+"_v0.csv")

P:/Commun/Sources_donnees_complementaires/generation/recode_scenario/sample_10000_20250924/train_v0.csv


In [13]:
file_name_finetune = "val"
df_val.iloc[nb_samples_from_val:(nb_samples_from_val+val_size),:][cols].to_csv(path_results + job_name_train +"/"+file_name_finetune+"_v0.csv")
print(path_results + job_name_train +"/"+file_name_finetune+"_v0.csv")

P:/Commun/Sources_donnees_complementaires/generation/recode_scenario/sample_10000_20250924/val_v0.csv


In [15]:
nb_samples_from_test = nb_samples_from_val+val_size
file_name_finetune = "test"
final_id_test = min((nb_samples_from_test+test_size),len(df_val))
print("Final test size = " + str(final_id_test - nb_samples_from_test))
df_val.iloc[nb_samples_from_test:final_id_test,:][cols].to_csv(path_results + job_name_train +"/"+file_name_finetune+"_v0.csv")
print(path_results + job_name_train +"/"+file_name_finetune+"_v0.csv")

Final test size = 1000
P:/Commun/Sources_donnees_complementaires/generation/recode_scenario/sample_10000_20250924/test_v0.csv


In [1]:
import json

with open(r"G:\commun\Donnees_partagees\Les travaux du DIM\ExperimenationCIM11\202512\FICHCOMP\2026-02-03_FICHCOMP.codeencim11.json") as f:
    d = json.load(f)

In [24]:
df

,numRss,numAdminSej,infosRum,idCodeur
0,100896040,103094462 20250701,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",81524
1,100882429,103124033,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",81524
2,100893678,103134050,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",81524
3,100881483,103161033 20250520,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",81524
4,100881949,103166674,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",81524
...,...,...,...,...
100,871105772,872390669,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",528636
101,880403964,882380343,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",528636
102,880403822,882385706,[{'comm': 'cGFzIHRyb3V2w6kgbGUgY29kZSByw6lwbGl...,528636
103,880410095,882397484 20250626,"[{'comm': '', 'contQual': False, 'dateEntreeRu...",528636


In [15]:
df = pd.json_normalize(d['enqCim11'])

In [20]:
df = df.assign(idCodeur = df.infosRum.apply(lambda x: pd.json_normalize(x[0])["idCodeur"]))

In [33]:
df.groupby('idCodeur').agg({'numRss':'size','numRss':'count'}).to_excel("C:/data/wd/test.xlsx")

In [25]:
df.groupby('idCodeur').numRss.nunique()

idCodeur
137784    12
153874    27
528636    15
57049     18
572463     6
72547      5
81524     22
Name: numRss, dtype: int64

In [29]:
df.groupby('idCodeur').size()

idCodeur
137784    12
153874    27
528636    15
57049     18
572463     6
72547      5
81524     22
dtype: int64

In [32]:
df.numRss.size

105